In [0]:
import mlflow

print("MLflow version:", mlflow.__version__)

In [0]:
mlflow.set_experiment("/Shared/kkbox-churn-model")

print("MLflow experiment configured successfully.")

In [0]:
from pyspark.ml.classification import RandomForestClassificationModel

model_path = "/Volumes/workspace/kkbox/models/churn_random_forest"

rf_model = RandomForestClassificationModel.load(model_path)

print("Random Forest loaded successfully.")
print("Number of trees:", rf_model.getNumTrees)

In [0]:
dbutils.fs.mkdirs(
    "/Volumes/workspace/kkbox/models/mlflow_tmp"
)

print("MLflow temporary directory ready.")

In [0]:
with mlflow.start_run(run_name="random_forest_final") as run:

    # Model parameters
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("num_trees", rf_model.getNumTrees)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("seed", 42)
    mlflow.log_param("training_split", 0.80)
    mlflow.log_param("test_split", 0.20)
    mlflow.log_param("class_weighting", "balanced")

    # Final held-out test metrics
    mlflow.log_metric("roc_auc", 0.9465)
    mlflow.log_metric("pr_auc", 0.5666)
    mlflow.log_metric("precision", 0.3196)
    mlflow.log_metric("recall", 0.9137)
    mlflow.log_metric("f1", 0.4736)

    # Log Spark model using a UC Volume as temporary storage
    mlflow.spark.log_model(
        rf_model,
        "random_forest_model",
        dfs_tmpdir="/Volumes/workspace/kkbox/models/mlflow_tmp"
    )

    run_id = run.info.run_id

print("MLflow run completed successfully.")
print("Run ID:", run_id)

In [0]:
with mlflow.start_run(run_name="logistic_regression_baseline") as run:

    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("max_iter", 50)
    mlflow.log_param("reg_param", 0.1)
    mlflow.log_param("elastic_net_param", 0.0)
    mlflow.log_param("training_split", 0.80)
    mlflow.log_param("test_split", 0.20)
    mlflow.log_param("class_weighting", "balanced")

    mlflow.log_metric("roc_auc", 0.8845)
    mlflow.log_metric("pr_auc", 0.3272)
    mlflow.log_metric("precision", 0.2373)
    mlflow.log_metric("recall", 0.7720)
    mlflow.log_metric("f1", 0.3630)

    print("Logistic Regression baseline logged.")
    print("Run ID:", run.info.run_id)

In [0]:
from pyspark.sql import functions as F

In [0]:
comparison_data = [
    (
        "Logistic Regression",
        0.8845,
        0.3272,
        0.2373,
        0.7720,
        0.3630
    ),
    (
        "Random Forest",
        0.9465,
        0.5666,
        0.3196,
        0.9137,
        0.4736
    )
]

comparison_df = spark.createDataFrame(
    comparison_data,
    [
        "model",
        "roc_auc",
        "pr_auc",
        "precision",
        "recall",
        "f1"
    ]
)

display(
    comparison_df.orderBy(F.desc("roc_auc"))
)

In [0]:
comparison_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.model_comparison")

print("Model comparison saved successfully.")

In [0]:
display(
    spark.table("workspace.kkbox.model_comparison")
        .orderBy(F.desc("roc_auc"))
)

In [0]:
import mlflow

experiment = mlflow.get_experiment_by_name("/Shared/kkbox-churn-model")

print("Experiment ID:", experiment.experiment_id)
print("Experiment name:", experiment.name)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"]
)

display(
    runs[
        [
            "run_id",
            "metrics.roc_auc",
            "metrics.pr_auc",
            "metrics.precision",
            "metrics.recall",
            "metrics.f1"
        ]
    ]
)